In [22]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("my_name").getOrCreate()

In [23]:
data = [

         (1,"amar","pune",29,"employed"),
         (2,"samar","mumbai",25,"unemployed"),
         (3,"ram","delhi",30,"employed"),
         (4,"shyam","bangalore",28,"unemployed"),
         (5,"gopal","chennai",32,"employed")
]

columns = ["id","name","city","age","status"]

df1 = spark.createDataFrame(data,columns)
df1.show()

employed_df = df1.filter(df1.status == "employed")
employed_df.show()
unemployed_df = df1.filter(df1.status == "unemployed")
unemployed_df.show()

+---+-----+---------+---+----------+
| id| name|     city|age|    status|
+---+-----+---------+---+----------+
|  1| amar|     pune| 29|  employed|
|  2|samar|   mumbai| 25|unemployed|
|  3|  ram|    delhi| 30|  employed|
|  4|shyam|bangalore| 28|unemployed|
|  5|gopal|  chennai| 32|  employed|
+---+-----+---------+---+----------+

+---+-----+-------+---+--------+
| id| name|   city|age|  status|
+---+-----+-------+---+--------+
|  1| amar|   pune| 29|employed|
|  3|  ram|  delhi| 30|employed|
|  5|gopal|chennai| 32|employed|
+---+-----+-------+---+--------+

+---+-----+---------+---+----------+
| id| name|     city|age|    status|
+---+-----+---------+---+----------+
|  2|samar|   mumbai| 25|unemployed|
|  4|shyam|bangalore| 28|unemployed|
+---+-----+---------+---+----------+



In [24]:
data = [

            (1,"amar","pune",29,90000),
            (2,"samar","mumbai",25,0),
            (3,"ram","delhi",30,120000),
            (4,"shyam","bangalore",28,0),
            (5,"gopal","chennai",32,150000)
]

columns = ["id","name","city","age","salary"]

df2 = spark.createDataFrame(data,columns)
df2.show()

+---+-----+---------+---+------+
| id| name|     city|age|salary|
+---+-----+---------+---+------+
|  1| amar|     pune| 29| 90000|
|  2|samar|   mumbai| 25|     0|
|  3|  ram|    delhi| 30|120000|
|  4|shyam|bangalore| 28|     0|
|  5|gopal|  chennai| 32|150000|
+---+-----+---------+---+------+



In [25]:
df2.show()
df1.show()

+---+-----+---------+---+------+
| id| name|     city|age|salary|
+---+-----+---------+---+------+
|  1| amar|     pune| 29| 90000|
|  2|samar|   mumbai| 25|     0|
|  3|  ram|    delhi| 30|120000|
|  4|shyam|bangalore| 28|     0|
|  5|gopal|  chennai| 32|150000|
+---+-----+---------+---+------+

+---+-----+---------+---+----------+
| id| name|     city|age|    status|
+---+-----+---------+---+----------+
|  1| amar|     pune| 29|  employed|
|  2|samar|   mumbai| 25|unemployed|
|  3|  ram|    delhi| 30|  employed|
|  4|shyam|bangalore| 28|unemployed|
|  5|gopal|  chennai| 32|  employed|
+---+-----+---------+---+----------+



In [26]:
#1. Find all unemployed employees
unemployed_employees = df1.filter(df1.status == "unemployed")
unemployed_employees.show()

#using sparksql
df1.createOrReplaceTempView("employees")
spark.sql(
    """ select * from employees
    where status = "unemployed" """
).show()

+---+-----+---------+---+----------+
| id| name|     city|age|    status|
+---+-----+---------+---+----------+
|  2|samar|   mumbai| 25|unemployed|
|  4|shyam|bangalore| 28|unemployed|
+---+-----+---------+---+----------+

+---+-----+---------+---+----------+
| id| name|     city|age|    status|
+---+-----+---------+---+----------+
|  2|samar|   mumbai| 25|unemployed|
|  4|shyam|bangalore| 28|unemployed|
+---+-----+---------+---+----------+



In [27]:
#2. Find employees whose salary is 0
employess_with_zero_sal = df2.filter(df2.salary == 0)
employess_with_zero_sal.show()

#usning sparksql
df2.createOrReplaceTempView("salary")

spark.sql(
    """select * from salary
    where salary = 0"""
).show()

+---+-----+---------+---+------+
| id| name|     city|age|salary|
+---+-----+---------+---+------+
|  2|samar|   mumbai| 25|     0|
|  4|shyam|bangalore| 28|     0|
+---+-----+---------+---+------+

+---+-----+---------+---+------+
| id| name|     city|age|salary|
+---+-----+---------+---+------+
|  2|samar|   mumbai| 25|     0|
|  4|shyam|bangalore| 28|     0|
+---+-----+---------+---+------+



In [28]:
#3. Find average salary of employed employees

from pyspark.sql.functions import avg

df2.join (df1,"id")\
   .filter(df1.status == "employed")\
   .agg(avg(df2.salary).alias("average_slary"))\
   .show()

#using sparksql
df2.createOrReplaceTempView("df2")
df1.createOrReplaceTempView("df1")

spark.sql(
    """select avg(salary) AS average_salary
    from df2
    join df1 on df2.id = df1.id
    where df1.status = "employed" """
).show()

+-------------+
|average_slary|
+-------------+
|     120000.0|
+-------------+

+--------------+
|average_salary|
+--------------+
|      120000.0|
+--------------+



In [31]:
#4. Find highest salary employee
df2.orderBy(df2.salary.desc()).limit(1).show()

#using sparksql
spark.sql(
    """select * from df2
    order by salary desc
    limit 1"""
).show()


+---+-----+-------+---+------+
| id| name|   city|age|salary|
+---+-----+-------+---+------+
|  5|gopal|chennai| 32|150000|
+---+-----+-------+---+------+

+---+-----+-------+---+------+
| id| name|   city|age|salary|
+---+-----+-------+---+------+
|  5|gopal|chennai| 32|150000|
+---+-----+-------+---+------+

